# Credit Scoring Project — Bank Loan Risk Classification

## Project Overview
Goal: Classify risk level for new loan applicants into 'Good' or 'Bad' risk using historical German Credit Data.

### Roadmap:
1. **Day 1:** Understand underwriting rules (Exploratory Data Analysis)
2. **Day 2:** Build feature pipeline and handle class imbalance (SMOTE)
3. **Day 3:** Train Advanced Models (CatBoost / XGBoost)
4. **Day 4:** Deployment Strategy (FastAPI Preparation)
5. **Day 5:** Executive Report


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

# Imbalanced Data & Advanced Models
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Visualization settings
plt.style.use('ggplot')
sns.set_palette("viridis")
%matplotlib inline

## Day 1 — Understanding Underwriting Rules (EDA)
In this phase, we analyze the factors that contribute to credit risk. Underwriting rules often look for stability (Job, Housing) and financial health (Savings, Checking account).

In [ ]:
df = pd.read_csv('german_credit_data.csv', index_col=0)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 12))

sns.countplot(x='Risk', data=df, ax=ax[0,0]).set_title('Risk Distribution (Imbalance)')
sns.histplot(x='Age', hue='Risk', data=df, kde=True, ax=ax[0,1]).set_title('Age vs Risk')
sns.boxplot(x='Risk', y='Credit amount', data=df, ax=ax[1,0]).set_title('Credit Amount vs Risk')
sns.countplot(x='Housing', hue='Risk', data=df, ax=ax[1,1]).set_title('Housing vs Risk')

plt.tight_layout()
plt.show()

## Day 2 — Feature Pipeline & Class Imbalance (SMOTE)
We handle missing values and prepare a pipeline that transforms numerical and categorical data, then applies SMOTE to balance the classes.

In [ ]:
# Handle Missing Values
df['Saving accounts'] = df['Saving accounts'].fillna('none')
df['Checking account'] = df['Checking account'].fillna('none')

# Target encoding
y = df['Risk'].map({'bad': 1, 'good': 0})
X = df.drop('Risk', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = ['Age', 'Credit amount', 'Duration']
categorical_features = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

print("Pipeline components defined. Ready for balancing.")

## Day 3 — Training Models (XGBoost & CatBoost)
We compare two state-of-the-art gradient boosting models.

In [ ]:
# XGBoost with SMOTE inside the pipeline
xgb_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))

# Save the model for Day 4
joblib.dump(xgb_pipeline, 'credit_model.pkl')
print("Model saved as credit_model.pkl")

## Day 4 — Deployment via FastAPI
The model is saved and ready for deployment. See `app.py` for the FastAPI implementation.

## Day 5 — Executive Report
### Key Findings:
- **Imbalance:** The dataset has more 'Good' than 'Bad' risks, which was addressed with SMOTE.
- **Key Drivers:** Credit Amount and Duration are strong predictors of high risk.
- **Performance:** The model achieves a balance between precision and recall, ensuring risky loans are flagged while minimizing loss of good customers.
